# Модуль 4c — microgpt: весь GPT в 200 строк

Здесь ты **прочитаешь и покрутишь** один файл Карпатого `microgpt.py` — это
полный GPT (обучение + attention + сэмплинг) на ~200 строках чистого Python.
Ноль зависимостей: никакого `pip install`, никакого PyTorch, всё на CPU.

Главная мысль, ради которой всё это: «настоящий» GPT — **не магия**. Это
обозримый код, который читается за вечер. Недосягаем не он, а масштаб данных
и вычислений.

Полный гайд с пояснениями — в
[README этой папки](https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/notebooks/module-4c-microgpt).
Лекция — [Модуль 4c](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04c-microgpt/).

Прогоняй сверху вниз. Места с `# TODO` — это домашка.

## Что нужно знать перед стартом (на пальцах)

Если ты прошёл [модуль 4a](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04a-micrograd/),
ты уже всё видел руками. Освежим четыре идеи без матана:

- **Value (autograd).** Обёртка вокруг числа, которая помнит, как считать
  градиент. Ты писал её в 4a. Здесь она **та же самая**, просто её много.
- **Forward.** Прогнали вход через слои слева направо — получили предсказание
  (вероятности следующей буквы).
- **Loss.** Одно число: насколько предсказание разошлось с правдой. Цель —
  гнать loss вниз.
- **Backward + шаг.** `loss.backward()` заполняет градиент каждого веса, потом
  оптимизатор крутит веса против градиента. Тот же цикл, что в 4a:
  `forward -> loss -> backward -> шаг`.

Новое в этом модуле — только **attention** (механизм, которым модель решает,
на какие предыдущие буквы посмотреть, чтобы предсказать следующую). Разберём
ниже, где он живёт в файле.

## Шаг 1 — скачиваем файл Карпатого

**Что:** один файл `microgpt.py` с гиста.

**Где:** публичный gist Карпатого.

**Как:** качаем через `urllib` (это stdlib, ничего ставить не надо).

**Зачем:** дальше мы его и прочитаем, и запустим прямо отсюда.

Если скачивание не прошло (например, корпоративный прокси) — открой ссылку из
ячейки в браузере, скопируй содержимое и сохрани руками в файл `microgpt.py`
рядом с ноутбуком.

In [ ]:
import os
import urllib.request

GIST_URL = (
    "https://gist.githubusercontent.com/karpathy/"
    "8627fe009c40f57531cb18360106ce95/raw/microgpt.py"
)

if not os.path.exists("microgpt.py"):
    urllib.request.urlretrieve(GIST_URL, "microgpt.py")
    print("скачал microgpt.py")
else:
    print("microgpt.py уже на месте")

print("размер файла:", os.path.getsize("microgpt.py"), "байт")

## Шаг 2 — прочитаем файл глазами (он короче, чем кажется)

**Что:** распечатаем сам файл с номерами строк, чтобы было куда «показывать
пальцем».

**Зачем:** домашка просит уметь указать, где `Value` (autograd из 4a), где
attention, где сэмплинг. Сначала посмотрим на код целиком, потом разметим
карту по номерам строк.

In [ ]:
with open("microgpt.py", encoding="utf-8") as f:
    lines = f.read().splitlines()

print("всего строк:", len(lines))
print("-" * 60)
for i, line in enumerate(lines, start=1):
    print(f"{i:3d} | {line}")

## Шаг 3 — карта файла: где что лежит

Та же логика, что в [лекции](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04c-microgpt/),
только привязанная к коду выше. Запусти ячейку — она найдёт ключевые строки и
распечатает их рядом с пояснением «что это и где ты это видел».

Читай так:

- `class Value` — **autograd из 4a**. Тот самый класс: forward строит граф,
  `.backward()` гонит градиенты назад. Здесь он чуть компактнее (локальные
  производные хранятся прямо в узле), но идея один в один.
- `def gpt(...)` — модель. Внутри неё **attention**: строки, где считаются
  `q`, `k`, `v` и `attn_weights = softmax(attn_logits)`. Это и есть «модель
  решает, на какие прошлые буквы посмотреть».
- цикл `for step in range(num_steps)` — **train loop**: forward -> loss ->
  `loss.backward()` -> шаг Adam. Тот же круг, что в 4a.
- блок `temperature` + `random.choices(...)` в конце — **сэмплинг**: из
  вероятностей тянем следующую букву.

In [ ]:
# Находим ориентиры по подстроке и печатаем номер строки.
# Если Карпатый поправит гист и что-то не найдётся - просто смотри глазами
# в распечатке из Шага 2, ориентиры подписаны словами.
landmarks = [
    ("class Value",        "autograd из 4a: обёртка над числом, что помнит градиент"),
    ("def backward",       "тот самый .backward() - топосорт + chain rule"),
    ("def gpt(",           "модель: эмбеддинги -> attention -> MLP -> логиты"),
    ("attn_weights",       "ATTENTION: на какие прошлые буквы смотреть (softmax)"),
    ("num_steps",          "сколько шагов обучения (ГИПЕРПАРАМЕТР для домашки)"),
    ("for step in range",  "train loop: forward -> loss -> backward -> шаг Adam"),
    ("loss.backward()",    "backprop по всем весам разом (как в 4a, только весов много)"),
    ("temperature =",      "СЭМПЛИНГ: 'творческость' выдачи (ГИПЕРПАРАМЕТР для домашки)"),
    ("random.choices",     "тянем следующую букву из вероятностей"),
]

for needle, note in landmarks:
    found = next((i for i, ln in enumerate(lines, start=1) if needle in ln), None)
    if found is None:
        print(f"  ??? | не нашёл '{needle}' - глянь глазами | {note}")
    else:
        print(f"стр {found:3d} | {needle:18s} | {note}")

## Шаг 4 — что именно произойдёт при запуске

Прежде чем нажать «выполнить», договоримся, чего ждать:

1. Если рядом нет `input.txt`, скрипт **сам скачает** датасет имён
   (`names.txt`, ~32 000 имён) и сохранит как `input.txt`. Это char-level
   датасет — словарь из отдельных букв.
2. Пойдёт обучение: `num_steps` шагов (по умолчанию 1000), на каждом печатается
   падающий `loss` (например, с ~3.3 вниз). На ноутбуке это пара минут —
   медленно именно потому, что autograd **скалярный** (считает по одному числу,
   как в 4a).
3. В конце скрипт сгенерирует **20 новых имён** — выдуманных, которых не было в
   датасете (`kamon`, `areli`, `jaire` — строчными, как в датасете). Модель
   выучила структуру (какие буквы за какими обычно идут) и «галлюцинирует»
   правдоподобные имена.

**Важно про скачивание датасета.** На Colab/Kaggle автоскачивание работает.
Локально на macOS иногда мешает SSL-сертификат Python — тогда датасет можно
положить руками: следующая ячейка скачает `input.txt` заранее и устойчиво.
Если ты на Colab — она тоже не помешает.

In [ ]:
# Подстелить соломку: заранее кладём input.txt, чтобы microgpt.py не качал его сам.
# Так обходим возможную проблему с SSL-сертификатом при локальном запуске.
NAMES_URL = "https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt"

if not os.path.exists("input.txt"):
    try:
        urllib.request.urlretrieve(NAMES_URL, "input.txt")
    except Exception as e:
        print("urllib не смог (вероятно SSL). Пробуем без проверки сертификата...")
        import ssl
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        with urllib.request.urlopen(NAMES_URL, context=ctx) as r:
            open("input.txt", "wb").write(r.read())
    print("input.txt готов")
else:
    print("input.txt уже на месте")

with open("input.txt", encoding="utf-8") as f:
    sample_names = [next(f).strip() for _ in range(5)]
print("первые имена в датасете:", sample_names)

## Шаг 5 — запускаем обучение (это и есть «тяжёлая» часть)

**Что:** запускаем `microgpt.py` как есть, через системный Python.

**Где:** прямо в ноутбуке через `!python microgpt.py` (в Colab/Kaggle
восклицательный знак выполняет shell-команду).

**Как долго:** пара минут на CPU при `num_steps=1000`. GPU **не нужен и не
ускорит** — autograd скалярный, тензоров тут нет.

**Зачем:** увидеть своими глазами падающий loss и 20 сгенерированных имён.
Это твой артефакт для домашки.

Если запускаешь **локально в Jupyter**, а не в Colab — `!python` тоже работает.
Если вдруг нет — открой терминал в этой папке и набери `python microgpt.py`.

In [ ]:
# ТЯЖЁЛАЯ ЯЧЕЙКА: обучение ~1000 шагов, пара минут на CPU.
# loss будет печататься и убывать, в конце - 20 новых имён.
!python microgpt.py

Получилось? Тогда выполнены все четыре пункта из лекции:

1. файл запустился без ошибок и без `pip install`;
2. loss печатался и убывал;
3. в конце вывелось ~20 имён, похожих на настоящие, но новых;
4. ты можешь показать в файле, где `Value` (autograd из 4a), где attention
   (внутри `gpt()`), а где сэмплинг — карту дал Шаг 3.

Это и есть закрытый модуль. Ниже — домашка-твики из лекции.

---
# Домашка — «Прочитать и покрутить»

Задания **совпадают** с блоком «Прочитать и покрутить» в
[лекции 4c](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04c-microgpt/).
Цель: убедиться, что файл — не магия, а понятный код, и связать его с тем, что
ты уже знаешь.

Ниже мы будем **править файл прямо из ноутбука** маленькой утилитой
`replace_in_file` — она меняет одну строку в `microgpt.py` и сохраняет.
Так не придётся открывать редактор.

In [ ]:
def replace_in_file(path, old, new):
    """Заменить первое вхождение old на new в файле. Падает, если не нашёл."""
    text = open(path, encoding="utf-8").read()
    if old not in text:
        raise ValueError(f"не нашёл в файле строку: {old!r}")
    text = text.replace(old, new, 1)
    open(path, "w", encoding="utf-8").write(text)
    print(f"заменил:\n  {old}\n->\n  {new}")

## ДЗ-1. Покрути `temperature` (сэмплинг)

**Что такое temperature на пальцах:** ручка «творческости» при генерации.
Низкая (`0.1`) — модель почти всегда берёт самую вероятную букву, имена
получаются скучные и однообразные. Высокая (`1.0`) — модель чаще рискует,
имена разнообразнее, но и «мусора» больше.

Найди в файле строку `temperature = 0.5` (она в блоке inference, в самом
конце). Прогони с `0.1`, потом с `1.0`, сравни выдачу.

**Связь с курсом:** это ровно та же ручка, что в шаге 4 модуля 5 («слышим
разницу»).

In [ ]:
# Ставим temperature = 0.1 и запускаем.
# Обучение снова пройдёт целиком - это нормально, пара минут.
replace_in_file("microgpt.py", "temperature = 0.5", "temperature = 0.1")
!python microgpt.py

In [ ]:
# Теперь temperature = 1.0. Возвращаем с 0.1 на 1.0 и сравниваем имена.
replace_in_file("microgpt.py", "temperature = 0.1", "temperature = 1.0")
!python microgpt.py

In [ ]:
# Вернём temperature к исходным 0.5, чтобы дальше было как в оригинале.
replace_in_file("microgpt.py", "temperature = 1.0", "temperature = 0.5")
print("temperature снова 0.5")

**TODO (ДЗ-1):** в одном предложении опиши, как изменились имена между
`temperature` 0.1 и 1.0, и свяжи это с шагом 4 модуля 5.

_Впиши ответ сюда, дважды кликнув по ячейке:_

> ...

## ДЗ-2. Покрути `num_steps` (число шагов обучения)

**Что это:** сколько раз модель сделает круг `forward -> loss -> backward ->
шаг`. Меньше шагов — модель недоучилась, имена хуже. Больше — дольше учится,
loss обычно ниже, имена правдоподобнее (до какого-то предела).

Найди `num_steps = 1000`. Поставь меньше (например, `200`) и больше (например,
`2000`). Смотри на финальный `loss` и на качество имён.

**Замечание про время:** `2000` шагов = примерно вдвое дольше. Если на CPU
долго — это и есть тот самый scalar autograd. GPU тут не помог бы.

In [ ]:
# Меньше шагов: 200. Модель недоучится - имена будут хуже, loss выше.
replace_in_file("microgpt.py", "num_steps = 1000", "num_steps = 200")
!python microgpt.py

In [ ]:
# Больше шагов: 2000 (дольше). Сравни финальный loss и имена с прогоном на 200.
replace_in_file("microgpt.py", "num_steps = 200", "num_steps = 2000")
!python microgpt.py

In [ ]:
# Вернём num_steps к 1000.
replace_in_file("microgpt.py", "num_steps = 2000", "num_steps = 1000")
print("num_steps снова 1000")

**TODO (ДЗ-2):** в одном предложении — что произошло с финальным `loss` и
качеством имён при 200 против 2000 шагов?

> ...

## ДЗ-3. Замени датасет (свой `input.txt`)

**Идея:** модель не знает, что учит имена. Она учит **структуру строк** из
файла `input.txt`. Подсунь свой список коротких слов — и она начнёт
«галлюцинировать» слова в твоём стиле.

Требования к файлу: список коротких слов, по одному на строку, **не меньше
500 штук** (иначе нечему учиться). Например: города, покемоны, имена котов,
названия команд.

Ниже два варианта: (а) сделать игрушечный датасет прямо в коде из списка, или
(б) если у тебя уже есть свой файл — просто положи его рядом под именем
`input.txt` и пропусти ячейку-генератор.

Для примера соберём датасет из синтетических «городов» (чтобы ноутбук был
самодостаточным). **Замени список `my_words` на свой** — это и есть суть
задания.

In [ ]:
# TODO (ДЗ-3): подставь СВОЙ список слов (>= 500 строк) вместо примера ниже.
# Пример: маленький генератор псевдо-слов, чтобы ноутбук был самодостаточным.
# Лучше замени его на реальный список (города/покемоны/коты/команды).
import random as _r
_r.seed(0)

starts = ["ka", "ro", "mi", "to", "sa", "len", "bri", "dor", "vel", "nor"]
mids   = ["la", "ri", "to", "na", "vi", "mo", "se", "da", "ko", ""]
ends   = ["n", "sk", "a", "ton", "ville", "grad", "by", "o", "in", "el"]

my_words = set()
while len(my_words) < 800:
    w = _r.choice(starts) + _r.choice(mids) + _r.choice(ends)
    my_words.add(w)
my_words = sorted(my_words)

# Сохраняем как input.txt (старый перезапишется - так и надо).
with open("input.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(my_words) + "\n")

print("слов в датасете:", len(my_words))
print("примеры:", my_words[:10])

In [ ]:
# Запускаем обучение на НОВОМ датасете.
# microgpt.py увидит уже лежащий input.txt и не станет качать имена.
# В конце - слова в стиле твоего списка.
!python microgpt.py

**TODO (ДЗ-3):** скопируй сюда 5 сгенерированных «слов» на своём `input.txt`.

> ...

## ДЗ-4. Покажи пальцем, где autograd

**TODO (ДЗ-4):** в одном предложении — какая строка/функция в `microgpt.py`
соответствует классу `Value`, который ты писал руками в модуле 4a? (Подсказка:
карта из Шага 3 уже распечатала номер строки.) Заодно объясни себе, почему
этот код не потянет Пушкина — упрётся в scalar autograd (по одному числу за
раз на CPU).

> ...

## Что сдать

Ровно как в [лекции](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04c-microgpt/):

- По три сэмпла имён на каждое значение `temperature` (0.1 и 1.0) + одно
  предложение, чем они отличаются (ДЗ-1).
- 5 сгенерированных «слов» на твоём собственном `input.txt` (ДЗ-3).
- Одно предложение: какая строка/функция в `microgpt.py` соответствует классу
  `Value` из модуля 4a (ДЗ-4).

**Критерий приёма:** ты можешь показать пальцем в файле, где autograd, где
attention (внутри `gpt()`), а где сэмплинг — и объяснить, почему этот код не
потянет Пушкина. Доказательство — скриншот вывода 20 имён + сэмплы на своём
`input.txt`.

В чат как `[Модуль 4c, ДЗ] {ссылка/скриншоты}`.

## Если хочется глубже

Разбор `microgpt.py` построчно на русском — [статья на Habr](https://habr.com/ru/articles/996404/).
Следующий шаг — [Модуль 5: свой GPT за час](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-own-gpt/)
(тот же алгоритм, но на PyTorch и GPU).